# ART-760 — Does the classifier beat chance?
## Monte Carlo simulation for empirical baseline estimation in imbalanced datasets

**How to cite:** *(add citation here)*

---

### Pipeline overview

| Cell | Module | Role |
|------|--------|------|
| 1 | module-00-setup | Packages, parameters, file resolution |
| 2 | module-01-synth-corpus | Synthetic corpus generation (bypass if Corpus exists) |
| 3 | module-02-read-validate | Corpus reading & validation |
| 4 | module-03-simulate | Monte Carlo simulation (bypass if Simulations exists) |
| 5 | *[OPTIONAL]* validation-known-cases | Known-outcome verification (run **instead of** module-03, then module-04) |
| 6 | module-04-report | Metric computation & reporting |
| — | *[OPTIONAL]* inspection | Per-iteration confusion matrix & metrics (run after module-04) |
| — | *[OPTIONAL]* inspection-save-excel | Save inspection table to Excel |
| — | *[OPTIONAL]* heterogeneous-classifiers | Heterogeneous reviewer simulation (replaces modules 01–03) |
| — | module-99-requirements | Dependency snapshot |

> **Normal execution:** run cells 1 → 6 in order, skipping the optional validation cell.  
> **Validation run:** run cells 1 → 4 (module-03), then validation-known-cases **instead of** module-03, then module-04.

### module-00-setup

In [1]:
# module-00-setup
# Installs and imports all required packages, defines all global parameters,
# and resolves the path to the working Excel file.
# All downstream modules consume EXCEL_PATH without modification.

# ******************************************
# DEPENDENCIES
# ******************************************
import importlib, subprocess, sys

def ensure_packages(pkgs: dict):
    """Install missing packages, then import all. pkgs = {import_name: pip_name}"""
    for imp, pip in pkgs.items():
        if importlib.util.find_spec(imp) is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pip])
    for imp in pkgs:
        globals()[imp] = importlib.import_module(imp)

ensure_packages({
    "pandas":   "pandas",    # data wrangling
    "openpyxl": "openpyxl",  # read/write Excel files (.xlsx)
    "numpy":    "numpy",     # numerical operations & random sampling
})

import random
import numpy as np
import pandas as pd
from pathlib import Path
# ******************************************

# ******************************************
# PARAMETERS
# ******************************************
EXCEL_PATH = None                    # "C:/full/path/to/file.xlsx"  |  None → resolve below
EXCEL_FILE = "ART760_templateBlank2.xlsx"  # filename in notebook directory  |  None → file picker

POS_LABELS = ["included"]   # label(s) mapped to POSITIVE — use list for multiple
NEG_LABELS = ["excluded"]   # label(s) mapped to NEGATIVE — use list for multiple
N_ITER     = 10_000         # number of Monte Carlo iterations
SEED       = 34             # global random seed  |  None → no seed

SH_SPEC = "Spec"
SH_CORP = "Corpus"
SH_SIMS = "Simulations"
SH_REP  = "Reporting"
# ******************************************

# ******************************************
# Global seed
# ******************************************
if SEED is not None:
    random.seed(SEED)
    np.random.seed(SEED)
# ******************************************

# ******************************************
# Resolve Excel path
# ******************************************
def _resolve_excel(excel_path, filename, notebook_dir):
    """
    Priority:
      1. EXCEL_PATH set and file exists  → use directly.
      2. EXCEL_FILE set and found in notebook_dir  → build full path.
      3. Neither / not found  → open tkinter file picker.
    """
    # 1. Full path provided
    if excel_path is not None and str(excel_path).strip():
        p = Path(excel_path)
        if not p.exists():
            raise FileNotFoundError(f"EXCEL_PATH not found: {excel_path}")
        print(f"✔ Using EXCEL_PATH: {p}")
        return p

    # 2. Filename provided
    if filename is not None and str(filename).strip():
        p = notebook_dir / filename
        if p.exists():
            print(f"✔ File found: {p.name}")
            return p
        print(f"⚠ '{filename}' not found in {notebook_dir}. Opening file picker…")

    # 3. File picker (works in both Jupyter and VS Code Jupyter kernel)
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    root.wm_attributes("-topmost", True)
    chosen = filedialog.askopenfilename(
        title="Select the scenario Excel file",
        initialdir=str(notebook_dir),
        filetypes=[("Excel files", "*.xlsx"), ("All files", "*.*")]
    )
    root.destroy()
    if not chosen:
        raise ValueError("No file selected.")
    p = Path(chosen)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {chosen}")
    print(f"✔ Selected: {p.name}")
    return p

# Resolve notebook directory: __file__ not available in notebooks,
# so fall back to Path.cwd() (which is the notebook directory in Jupyter).
_notebook_dir = Path.cwd()

EXCEL_PATH = _resolve_excel(EXCEL_PATH, EXCEL_FILE, _notebook_dir)
print(f"   Working directory: {_notebook_dir}")


✔ File found: ART760_templateBlank2.xlsx
   Working directory: m:\OneDrive - UPV\R\ART-760


### module-01-synth-corpus

In [2]:
# module-01-synth-corpus
# Generates a synthetic labelled corpus from the Spec sheet and writes it to
# the Corpus sheet. Automatically skipped if Corpus already contains data.
# Depends on: EXCEL_PATH, SH_CORP, SH_SPEC, SEED, POS_LABELS, NEG_LABELS (module-00)

# ******************************************
# Bypass guard — skip if Corpus already has data
# ******************************************
import openpyxl

def _read_sheet_safe(path, sheet_name):
    """Return DataFrame for sheet_name, or empty DataFrame if sheet missing/empty."""
    try:
        df = pd.read_excel(path, sheet_name=sheet_name, engine="openpyxl")
        return df
    except Exception:
        return pd.DataFrame()

_corpus_existing = _read_sheet_safe(EXCEL_PATH, SH_CORP)
_skip_m1 = len(_corpus_existing) > 0

if _skip_m1:
    print(f"⚠ Module 1 skipped: Corpus sheet already contains "
          f"{len(_corpus_existing)} rows. Proceed to Module 2.")
    print("  Label distribution in existing corpus:")
    print(_corpus_existing["Label"].value_counts().to_string())

del _corpus_existing
# ******************************************

if not _skip_m1:

    # ******************************************
    # Read Spec
    # ******************************************
    spec_raw = pd.read_excel(EXCEL_PATH, sheet_name=SH_SPEC, engine="openpyxl")

    required_spec = {"n_items", "Label", "Proportion"}
    missing_spec  = required_spec - set(spec_raw.columns)
    if missing_spec:
        raise ValueError(f"Spec sheet missing columns: {', '.join(missing_spec)}")

    spec_raw = spec_raw.dropna(subset=["Label", "Proportion"])

    n_items  = int(spec_raw["n_items"].iloc[0])
    levels_k = spec_raw["Label"].tolist()
    probs_raw = spec_raw["Proportion"].to_numpy(dtype=float)
    probs_k  = probs_raw / probs_raw.sum()   # renormalise

    if abs(probs_raw.sum() - 1.0) > 0.01:
        print("⚠ Spec proportions did not sum to 1 — renormalised automatically.")

    print(f"Spec loaded: {n_items} items | {len(levels_k)} labels: "
          + " | ".join(f"{l}={p:.3f}" for l, p in zip(levels_k, probs_k)))
    # ******************************************

    # ******************************************
    # Generate synthetic corpus
    # ******************************************
    rng = np.random.default_rng(SEED)

    counts = np.round(probs_k * n_items).astype(int)
    counts[0] = n_items - counts[1:].sum()   # adjust first to guarantee exact total

    label_vec = np.repeat(levels_k, counts)
    rng.shuffle(label_vec)

    corpus_synth = pd.DataFrame({
        "ID":       np.arange(1, n_items + 1),
        "Title":    [f"Synthetic reference {i}" for i in range(1, n_items + 1)],
        "Abstract": [f"Abstract placeholder for item {i}" for i in range(1, n_items + 1)],
        "Label":    label_vec,
    })

    print("Synthetic corpus generated:")
    print(corpus_synth["Label"].value_counts().to_string())
    # ******************************************

    # ******************************************
    # Write to Excel (_synthetic.xlsx)
    # ******************************************
    synth_path = EXCEL_PATH.parent / (EXCEL_PATH.stem + "_synthetic.xlsx")

    # Copy workbook then replace/add Corpus sheet
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Border, Side

    wb = load_workbook(EXCEL_PATH)
    if SH_CORP in wb.sheetnames:
        del wb[SH_CORP]
    ws = wb.create_sheet(SH_CORP)

    # Write header with style
    header_fill = PatternFill("solid", fgColor="E2EFDA")
    header_font = Font(bold=True)
    for col_idx, col_name in enumerate(corpus_synth.columns, start=1):
        cell = ws.cell(row=1, column=col_idx, value=col_name)
        cell.fill = header_fill
        cell.font = header_font

    # Write data rows
    for row_idx, row in enumerate(corpus_synth.itertuples(index=False), start=2):
        for col_idx, value in enumerate(row, start=1):
            ws.cell(row=row_idx, column=col_idx, value=value)

    wb.save(synth_path)
    EXCEL_PATH = synth_path
    print(f"✔ Synthetic corpus written to '{EXCEL_PATH.name}'")
    # ******************************************


Spec loaded: 130 items | 2 labels: included=0.200 | excluded=0.800
Synthetic corpus generated:
Label
excluded    104
included     26
✔ Synthetic corpus written to 'ART760_templateBlank2_synthetic.xlsx'


### module-02-read-validate

In [3]:
# module-02-read-validate
# Reads and validates the Corpus and Spec sheets. Binarises labels into
# positive / negative. Outputs: corpus_df, spec_df, n_items, levels_k, probs_k.
# Depends on: EXCEL_PATH, SH_CORP, SH_SPEC, POS_LABELS, NEG_LABELS (module-00)

# ******************************************
# Read Corpus
# ******************************************
corpus_raw = pd.read_excel(EXCEL_PATH, sheet_name=SH_CORP, engine="openpyxl")

required_cols = {"ID", "Label"}
optional_cols = {"Title", "Abstract"}

missing_req = required_cols - set(corpus_raw.columns)
if missing_req:
    raise ValueError(f"Corpus sheet missing required columns: {', '.join(missing_req)}")

missing_opt = optional_cols - set(corpus_raw.columns)
if missing_opt:
    print(f"⚠ Corpus sheet missing optional columns: {', '.join(missing_opt)}")

print(f"Corpus loaded: {len(corpus_raw)} rows | {len(corpus_raw.columns)} columns")
# ******************************************

# ******************************************
# Binarise labels
# ******************************************
unknown_labels = set(corpus_raw["Label"].unique()) - set(POS_LABELS) - set(NEG_LABELS)
if unknown_labels:
    print(f"⚠ Unknown label(s) — will be set to NA in label_bin: "
          f"{', '.join(str(l) for l in unknown_labels)}")

def _binarise_series(series, pos, neg):
    """Map label series to 'positive' / 'negative' / NaN."""
    return series.map(
        lambda x: "positive" if x in pos else ("negative" if x in neg else None)
    )

corpus_df = corpus_raw.copy()
corpus_df["label_bin"] = _binarise_series(corpus_df["Label"], POS_LABELS, NEG_LABELS)

n_pos = (corpus_df["label_bin"] == "positive").sum()
n_neg = (corpus_df["label_bin"] == "negative").sum()
n_na  = corpus_df["label_bin"].isna().sum()

print(f"Label binarisation: {n_pos} positive | {n_neg} negative | {n_na} NA")
# ******************************************

# ******************************************
# Read Spec
# ******************************************
spec_df = pd.read_excel(EXCEL_PATH, sheet_name=SH_SPEC, engine="openpyxl")

required_spec = {"n_items", "Label", "Proportion"}
missing_spec  = required_spec - set(spec_df.columns)
if missing_spec:
    raise ValueError(f"Spec sheet missing columns: {', '.join(missing_spec)}")

spec_df  = spec_df.dropna(subset=["Label", "Proportion"])
n_items  = int(spec_df["n_items"].iloc[0])
levels_k = spec_df["Label"].tolist()
probs_raw = spec_df["Proportion"].to_numpy(dtype=float)
probs_k  = probs_raw / probs_raw.sum()

if abs(probs_raw.sum() - 1.0) > 0.01:
    print("⚠ Spec proportions did not sum to 1 — renormalised automatically.")

print(f"Spec loaded: {len(levels_k)} label levels | "
      + " | ".join(f"{l}={p:.3f}" for l, p in zip(levels_k, probs_k)))
# ******************************************

# ******************************************
# Corpus summary
# ******************************************
print("─── Corpus summary " + "─" * 30)
print(f"  Total items   : {len(corpus_df)}")
print(f"  Positive (n)  : {n_pos} ({n_pos / len(corpus_df) * 100:.1f}%)")
print(f"  Negative (n)  : {n_neg} ({n_neg / len(corpus_df) * 100:.1f}%)")
print(f"  NA label_bin  : {n_na}")
print(f"  Imbalance ratio (neg/pos): {n_neg / n_pos:.2f}" if n_pos > 0
      else "  Imbalance ratio: undefined (no positives)")
print("─" * 50)
# ******************************************


Corpus loaded: 130 rows | 4 columns
Label binarisation: 26 positive | 104 negative | 0 NA
Spec loaded: 2 label levels | included=0.200 | excluded=0.800
─── Corpus summary ──────────────────────────────
  Total items   : 130
  Positive (n)  : 26 (20.0%)
  Negative (n)  : 104 (80.0%)
  NA label_bin  : 0
  Imbalance ratio (neg/pos): 4.00
──────────────────────────────────────────────────


### module-03-simulate

In [4]:
# module-03-simulate
# Generates N_ITER simulated classification vectors and writes them to the
# Simulations sheet. Bypass: if the sheet already contains data, loads it as
# sims_df and skips simulation — Module 4 can then run directly.
# Depends on: corpus_df, n_items, levels_k, probs_k (module-02)
#             EXCEL_PATH, SH_SIMS, N_ITER, SEED (module-00)

# ******************************************
# Bypass guard — skip if Simulations already has data
# ******************************************
_sims_existing = _read_sheet_safe(EXCEL_PATH, SH_SIMS)
_skip_m3 = len(_sims_existing) > 0

if _skip_m3:
    sims_df = _sims_existing
    print(f"⚠ Module 3 skipped: Simulations sheet already contains "
          f"{len(sims_df)} rows × {len(sims_df.columns) - 1} iterations. "
          f"Loaded as sims_df — proceed to Module 4.")
    del _sims_existing
# ******************************************

if not _skip_m3:
    del _sims_existing

    # ******************************************
    # Run simulation
    # ******************************************
    print(f"Running {N_ITER:,} iterations × {n_items:,} items…")

    rng = np.random.default_rng(SEED)

    # Draw all labels in one vectorised call: shape (n_items, N_ITER)
    _sim_matrix = rng.choice(levels_k, size=(n_items, N_ITER), p=probs_k)

    col_names = [f"iter_{i:04d}" for i in range(1, N_ITER + 1)]
    sims_df = pd.DataFrame(_sim_matrix, columns=col_names)
    sims_df.insert(0, "ID", corpus_df["ID"].values)
    del _sim_matrix

    print(f"✔ Simulation complete: {len(sims_df)} items × {N_ITER} iterations")
    # ******************************************

    # ******************************************
    # Write Simulations to Excel
    # ******************************************
    print("Writing to Excel…")

    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill

    wb = load_workbook(EXCEL_PATH)
    if SH_SIMS in wb.sheetnames:
        del wb[SH_SIMS]
    ws = wb.create_sheet(SH_SIMS)

    header_fill = PatternFill("solid", fgColor="FFF2CC")
    header_font = Font(bold=True)
    for col_idx, col_name in enumerate(sims_df.columns, start=1):
        cell = ws.cell(row=1, column=col_idx, value=col_name)
        cell.fill = header_fill
        cell.font = header_font

    for row_idx, row in enumerate(sims_df.itertuples(index=False), start=2):
        for col_idx, value in enumerate(row, start=1):
            ws.cell(row=row_idx, column=col_idx, value=value)

    wb.save(EXCEL_PATH)
    print(f"✔ Simulations written to sheet '{SH_SIMS}' in {EXCEL_PATH.name}")
    # ******************************************


Running 10,000 iterations × 130 items…
✔ Simulation complete: 130 items × 10000 iterations
Writing to Excel…
✔ Simulations written to sheet 'Simulations' in ART760_templateBlank2_synthetic.xlsx


### [OPTIONAL] validation-known-cases

> Run this cell **instead of module-03** if you want to verify that module-04
> computes metrics correctly. Skip it to run the normal Monte Carlo pipeline.
>
> Expected outcomes:
> | Vector | Recall | Specificity | Precision | MCC |
> |--------|--------|-------------|-----------|-----|
> | all_positive | 1 | 0 | prevalence | 0 |
> | all_negative | 0 | 1 | NA | 0 |
> | perfect | 1 | 1 | 1 | 1 |
> | random_fair | ≈0.5 | ≈0.5 | ≈prevalence | ≈0 |

In [12]:
# validation-known-cases  [OPTIONAL — run INSTEAD of module-03, then run module-04]
# Builds 4 known-outcome classification vectors, writes them to the
# Simulations sheet, and exposes them as sims_df for Module 4.
# Depends on: corpus_df (module-02) | EXCEL_PATH, SH_SIMS, POS_LABELS, NEG_LABELS, SEED (module-00)

# ******************************************
# Build validation vectors
# ******************************************
rng_val = np.random.default_rng(SEED)
n = len(corpus_df)

val_df = pd.DataFrame({
    "ID":           corpus_df["ID"].values,
    "all_positive": POS_LABELS[0],
    "all_negative": NEG_LABELS[0],
    "perfect":      corpus_df["Label"].values,
    "random_fair":  rng_val.choice(
                        [POS_LABELS[0], NEG_LABELS[0]],
                        size=n, p=[0.5, 0.5]
                    ),
})

print(f"Validation dataset built: {n} items × 4 cases")
print(f"  all_positive : all items classified as '{POS_LABELS[0]}'")
print(f"  all_negative : all items classified as '{NEG_LABELS[0]}'")
print(f"  perfect      : identical to gold standard")
print(f"  random_fair  : random 50/50 regardless of prevalence")
# ******************************************

# ******************************************
# Write to Excel (_validation.xlsx) & expose as sims_df
# ******************************************
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill

val_path = EXCEL_PATH.parent / (EXCEL_PATH.stem + "_validation.xlsx")

wb = load_workbook(EXCEL_PATH)
if SH_SIMS in wb.sheetnames:
    del wb[SH_SIMS]
ws = wb.create_sheet(SH_SIMS)

header_fill = PatternFill("solid", fgColor="FFE0CC")
header_font = Font(bold=True)
for col_idx, col_name in enumerate(val_df.columns, start=1):
    cell = ws.cell(row=1, column=col_idx, value=col_name)
    cell.fill = header_fill
    cell.font = header_font

for row_idx, row in enumerate(val_df.itertuples(index=False), start=2):
    for col_idx, value in enumerate(row, start=1):
        ws.cell(row=row_idx, column=col_idx, value=value)

wb.save(val_path)
EXCEL_PATH = val_path

sims_df = val_df   # expose for Module 4

print(f"✔ Validation cases written to '{EXCEL_PATH.name}' — original preserved")
print("  → Now run module-04 to verify metrics")
# ******************************************

Validation dataset built: 130 items × 4 cases
  all_positive : all items classified as 'included'
  all_negative : all items classified as 'excluded'
  perfect      : identical to gold standard
  random_fair  : random 50/50 regardless of prevalence
✔ Validation cases written to 'ART760_templateBlank2_hetero_validation.xlsx' — original preserved
  → Now run module-04 to verify metrics


### module-04-report

In [13]:
# module-04-report
# Computes 8 classification metrics (Recall, Specificity, Precision, NPV,
# F1, F2, Balanced Accuracy, MCC) for each iteration in sims_df.
# Summarises empirical distributions and writes results to the Reporting sheet.
# Depends on: corpus_df, sims_df (module-02 / module-03 or validation cell)
#             EXCEL_PATH, SH_REP, POS_LABELS, NEG_LABELS (module-00)

# ******************************************
# Helper functions
# ******************************************
def _binarise(series, pos, neg):
    """Map a label series to 'positive' / 'negative' / None."""
    return series.map(lambda x: "positive" if x in pos
                                else ("negative" if x in neg else None))

def _confusion(pred, truth):
    """Return TP, TN, FP, FN counts given two aligned Series."""
    tp = ((pred == "positive") & (truth == "positive")).sum()
    tn = ((pred == "negative") & (truth == "negative")).sum()
    fp = ((pred == "positive") & (truth == "negative")).sum()
    fn = ((pred == "negative") & (truth == "positive")).sum()
    return int(tp), int(tn), int(fp), int(fn)

def _metrics(tp, tn, fp, fn):
    """
    Compute 8 metrics from confusion matrix counts.
    Degenerate denominators (zero) → None (stored as NaN in DataFrame).
    MCC denominator uses float to avoid integer overflow on large corpora.
    """
    recall   = tp / (tp + fn)          if (tp + fn) > 0 else None
    spec     = tn / (tn + fp)          if (tn + fp) > 0 else None
    prec     = tp / (tp + fp)          if (tp + fp) > 0 else None
    npv      = tn / (tn + fn)          if (tn + fn) > 0 else None
    f1       = (2 * prec * recall / (prec + recall)
                if prec is not None and recall is not None
                   and (prec + recall) > 0 else None)
    f2       = (5 * prec * recall / (4 * prec + recall)
                if prec is not None and recall is not None
                   and (4 * prec + recall) > 0 else None)
    bal_acc  = ((recall + spec) / 2
                if recall is not None and spec is not None else None)
    mcc_den  = ((float(tp + fp) * float(tp + fn)
                 * float(tn + fp) * float(tn + fn)) ** 0.5)
    mcc      = (tp * tn - fp * fn) / mcc_den if mcc_den > 0 else None
    return {
        "Recall": recall, "Specificity": spec, "Precision": prec, "NPV": npv,
        "F1": f1, "F2": f2, "BalancedAccuracy": bal_acc, "MCC": mcc,
    }
# ******************************************

# ******************************************
# Compute metrics per iteration
# ******************************************
truth     = corpus_df["label_bin"]
iter_cols = [c for c in sims_df.columns if c != "ID"]

print(f"Computing metrics for {len(iter_cols):,} iterations…")

rows = []
for col in iter_cols:
    pred = _binarise(sims_df[col], POS_LABELS, NEG_LABELS)
    tp, tn, fp, fn = _confusion(pred, truth)
    rows.append(_metrics(tp, tn, fp, fn))

metrics_mat = pd.DataFrame(rows, index=iter_cols)
print(f"✔ Metrics computed: {len(metrics_mat)} iterations × {len(metrics_mat.columns)} metrics")
# ******************************************

# ******************************************
# Summarise empirical distributions
# ******************************************
metric_names = ["Recall", "Specificity", "Precision", "NPV",
                "F1", "F2", "BalancedAccuracy", "MCC"]

summary_rows = []
for m in metric_names:
    vals = metrics_mat[m].dropna()
    na_pct = round(metrics_mat[m].isna().mean() * 100, 1)
    summary_rows.append({
        "Metric":   m,
        "Mean":     vals.mean(),
        "Median":   vals.median(),
        "SD":       vals.std(),
        "CI95_Lo":  vals.quantile(0.025),
        "CI95_Hi":  vals.quantile(0.975),
        "NA_pct":   na_pct,
    })

report_df = pd.DataFrame(summary_rows)

print("─── Empirical baseline summary " + "─" * 19)
print(report_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("─" * 50)
# ******************************************

# ******************************************
# Write Reporting to Excel
# ******************************************
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, numbers

wb = load_workbook(EXCEL_PATH)
if SH_REP in wb.sheetnames:
    del wb[SH_REP]
ws = wb.create_sheet(SH_REP)

header_fill = PatternFill("solid", fgColor="DCE6F1")
header_font = Font(bold=True)
for col_idx, col_name in enumerate(report_df.columns, start=1):
    cell = ws.cell(row=1, column=col_idx, value=col_name)
    cell.fill = header_fill
    cell.font = header_font

num_fmt = "0.0000"
for row_idx, row in enumerate(report_df.itertuples(index=False), start=2):
    for col_idx, value in enumerate(row, start=1):
        cell = ws.cell(row=row_idx, column=col_idx, value=value)
        if col_idx > 1 and col_idx < len(report_df.columns):  # numeric cols, not NA_pct
            cell.number_format = num_fmt

wb.save(EXCEL_PATH)
print(f"✔ Report written to sheet '{SH_REP}' in {EXCEL_PATH.name}")
# ******************************************


Computing metrics for 4 iterations…
✔ Metrics computed: 4 iterations × 8 metrics
─── Empirical baseline summary ───────────────────
          Metric   Mean  Median     SD  CI95_Lo  CI95_Hi  NA_pct
          Recall 0.6442  0.7885 0.4735   0.0433   1.0000  0.0000
     Specificity 0.6394  0.7788 0.4745   0.0418   1.0000  0.0000
       Precision 0.4820  0.2459 0.4492   0.2023   0.9623 25.0000
             NPV 0.8802  0.8406 0.1057   0.8020   0.9920 25.0000
              F1 0.5594  0.3448 0.3816   0.3339   0.9672 25.0000
              F2 0.6700  0.5556 0.2902   0.4596   0.9778 25.0000
BalancedAccuracy 0.6418  0.5337 0.2409   0.5000   0.9675  0.0000
             MCC 0.5539  0.5539 0.6308   0.1302   0.9777 50.0000
──────────────────────────────────────────────────
✔ Report written to sheet 'Reporting' in ART760_templateBlank2_hetero_validation.xlsx


---
### [OPTIONAL] inspection

> Displays the confusion matrix (TP, TN, FP, FN) and all 8 metrics for each
> iteration. Runs on `metrics_mat` and `sims_df` produced by the most recent
> module-04 execution.


In [14]:
# inspection  [OPTIONAL — run after module-04]
# Displays per-iteration confusion matrix and 8 metrics.
# Depends on: corpus_df, sims_df, metrics_mat (module-04)
#             POS_LABELS, NEG_LABELS (module-00)

# ******************************************
# Build per-iteration inspection table
# ******************************************
truth_insp = corpus_df["label_bin"]
iter_cols_insp = [c for c in sims_df.columns if c != "ID"]

insp_rows = []
for col in iter_cols_insp:
    pred = _binarise(sims_df[col], POS_LABELS, NEG_LABELS)
    tp, tn, fp, fn = _confusion(pred, truth_insp)
    row = {"Classifier": col, "TP": tp, "TN": tn, "FP": fp, "FN": fn}
    row.update(metrics_mat.loc[col].to_dict())
    insp_rows.append(row)

inspection_df = pd.DataFrame(insp_rows)
# ******************************************

# ******************************************
# Display
# ******************************************
print(f"─── Per-iteration inspection ({len(inspection_df)} rows) " + "─" * 10)
_display_cols = ["Classifier", "TP", "TN", "FP", "FN"] + [
    "Recall", "Specificity", "Precision", "NPV",
    "F1", "F2", "BalancedAccuracy", "MCC"
]
print(
    inspection_df[_display_cols]
    .to_string(index=False, float_format=lambda x: f"{x:.4f}")
)
print("─" * 50)
# ******************************************


─── Per-iteration inspection (4 rows) ──────────
  Classifier  TP  TN  FP  FN  Recall  Specificity  Precision    NPV     F1     F2  BalancedAccuracy    MCC
all_positive  26   0 104   0  1.0000       0.0000     0.2000    NaN 0.3333 0.5556            0.5000    NaN
all_negative   0 104   0  26  0.0000       1.0000        NaN 0.8000    NaN    NaN            0.5000    NaN
     perfect  26 104   0   0  1.0000       1.0000     1.0000 1.0000 1.0000 1.0000            1.0000 1.0000
 random_fair  15  58  46  11  0.5769       0.5577     0.2459 0.8406 0.3448 0.4545            0.5673 0.1079
──────────────────────────────────────────────────


---
### [OPTIONAL] inspection-save-excel

> Saves the inspection table produced by the **inspection** cell to an
> `Inspection` sheet in the Excel file. Run only if you want to persist the
> per-iteration detail.


In [15]:
# inspection-save-excel  [OPTIONAL — run after inspection cell]
# Writes inspection_df to the Inspection sheet in the Excel file.
# Depends on: inspection_df (inspection cell) | EXCEL_PATH (module-00)

# ******************************************
# Write Inspection to Excel
# ******************************************
SH_INS = "Inspection"

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill

wb = load_workbook(EXCEL_PATH)
if SH_INS in wb.sheetnames:
    del wb[SH_INS]
ws = wb.create_sheet(SH_INS)

header_fill = PatternFill("solid", fgColor="EAD1DC")
header_font = Font(bold=True)
for col_idx, col_name in enumerate(inspection_df.columns, start=1):
    cell = ws.cell(row=1, column=col_idx, value=col_name)
    cell.fill = header_fill
    cell.font = header_font

num_fmt = "0.0000"
metric_cols_idx = {
    col: idx for idx, col in enumerate(inspection_df.columns, start=1)
    if col in {"Recall", "Specificity", "Precision", "NPV",
               "F1", "F2", "BalancedAccuracy", "MCC"}
}
for row_idx, row in enumerate(inspection_df.itertuples(index=False), start=2):
    for col_idx, value in enumerate(row, start=1):
        cell = ws.cell(row=row_idx, column=col_idx, value=value)
        if col_idx in metric_cols_idx.values():
            cell.number_format = num_fmt

wb.save(EXCEL_PATH)
print(f"✔ Inspection table written to sheet '{SH_INS}' in {EXCEL_PATH.name}")
# ******************************************


✔ Inspection table written to sheet 'Inspection' in ART760_templateBlank2_hetero_validation.xlsx


---
### [OPTIONAL] heterogeneous-classifiers

> Generates a synthetic dataset of heterogeneous classifiers simulating human
> reviewers with different labelling biases. Replaces the Simulations sheet so
> **module-04** can be run directly after this cell (modules 01–03 will auto-bypass).
>
> Results are saved to a new file with suffix `_hetero.xlsx` to preserve the original.


In [8]:
# heterogeneous-classifiers  [OPTIONAL]
# Generates N_CLASSIFIERS synthetic classifiers with heterogeneous labelling
# biases. Saves to [original]_hetero.xlsx and updates EXCEL_PATH.
# After this cell: run module-04 directly.
# Depends on: EXCEL_PATH, POS_LABELS, NEG_LABELS, SEED (module-00)

# ******************************************
# PARAMETERS
# ******************************************
N_CLASSIFIERS = 20

PROFILE = "fixed"
# "fixed"  — cycle through 6 predefined bias profiles
#            (lenient, strict, random, prevalence, biased_pos, biased_neg)
# "random" — each classifier gets a p_pos drawn uniformly from P_POS_RANGE

P_POS_RANGE = (0.05, 0.95)
# Only used when PROFILE == "random".
# Range from which each classifier's p_pos is drawn.
# ******************************************

# ******************************************
# Corpus — load or generate from Spec
# ******************************************
# Sanity-check: if corpus_df is already in the namespace, verify it matches Spec.
_spec_check = _read_sheet_safe(EXCEL_PATH, SH_SPEC)
if "corpus_df" in dir() and len(_spec_check) > 0:
    _spec_check = _spec_check.dropna(subset=["Label", "Proportion"])
    _expected_n = int(_spec_check["n_items"].iloc[0])
    if len(corpus_df) != _expected_n:
        print(f"⚠ corpus_df has {len(corpus_df)} rows but Spec expects "
              f"{_expected_n}. Reloading from Corpus sheet.")
        del corpus_df
    else:
        print(f"✔ corpus_df matches Spec: {len(corpus_df)} rows — reusing.")

_corpus_raw_h = _read_sheet_safe(EXCEL_PATH, SH_CORP)

if len(_corpus_raw_h) > 0:
    print(f"Corpus sheet found: {len(_corpus_raw_h)} rows — loaded as-is.")
    corpus_df = _corpus_raw_h.copy()
    corpus_df["label_bin"] = _binarise_series(
        corpus_df["Label"], POS_LABELS, NEG_LABELS
    )
else:
    print("Corpus sheet empty — generating from Spec…")
    _spec_h = pd.read_excel(EXCEL_PATH, sheet_name=SH_SPEC, engine="openpyxl")
    _spec_h  = _spec_h.dropna(subset=["Label", "Proportion"])
    _n       = int(_spec_h["n_items"].iloc[0])
    _lvls    = _spec_h["Label"].tolist()
    _probs   = _spec_h["Proportion"].to_numpy(dtype=float)
    _probs  /= _probs.sum()

    rng_h = np.random.default_rng(SEED)
    _counts    = np.round(_probs * _n).astype(int)
    _counts[0] = _n - _counts[1:].sum()
    _lvec = np.repeat(_lvls, _counts)
    rng_h.shuffle(_lvec)

    corpus_df = pd.DataFrame({
        "ID":       np.arange(1, _n + 1),
        "Title":    [f"Synthetic reference {i}" for i in range(1, _n + 1)],
        "Abstract": [f"Abstract placeholder for item {i}" for i in range(1, _n + 1)],
        "Label":    _lvec,
    })
    corpus_df["label_bin"] = _binarise_series(
        corpus_df["Label"], POS_LABELS, NEG_LABELS
    )

n_items_h  = len(corpus_df)
prevalence = (corpus_df["label_bin"] == "positive").mean()
print(f"Corpus ready: {n_items_h} items | prevalence = {prevalence:.3f}")
del _corpus_raw_h
# ******************************************

# ******************************************
# Classifier profiles
# ******************************************
_fixed_profiles = pd.DataFrame({
    "profile": ["lenient", "strict", "random",
                "prevalence", "biased_pos", "biased_neg"],
    "p_pos":   [0.75, 0.25, 0.50, prevalence, 0.90, 0.10],
})

rng_cls = np.random.default_rng(SEED)

if PROFILE == "fixed":
    _idx      = [(i % len(_fixed_profiles)) for i in range(N_CLASSIFIERS)]
    _sel      = _fixed_profiles.iloc[_idx].reset_index(drop=True)
    p_pos_vec = _sel["p_pos"].tolist()
    label_vec_cls = _sel["profile"].tolist()
else:
    p_pos_vec     = rng_cls.uniform(P_POS_RANGE[0], P_POS_RANGE[1],
                                    size=N_CLASSIFIERS).tolist()
    label_vec_cls = [f"random_{i:02d}" for i in range(1, N_CLASSIFIERS + 1)]

col_names_cls = [f"{lbl}_{i:02d}"
                 for i, lbl in enumerate(label_vec_cls, start=1)]

print("Classifier profiles assigned:")
for nm, p in zip(col_names_cls, p_pos_vec):
    print(f"  {nm:<25}  p_pos={p:.3f}")
# ******************************************

# ******************************************
# Generate classifications
# ******************************************
def _classify_one(p_pos, n, rng):
    return rng.choice(
        [POS_LABELS[0], NEG_LABELS[0]],
        size=n, p=[p_pos, 1 - p_pos]
    )

sims_dict = {"ID": corpus_df["ID"].values}
for nm, p in zip(col_names_cls, p_pos_vec):
    sims_dict[nm] = _classify_one(p, n_items_h, rng_cls)

sims_df = pd.DataFrame(sims_dict)

print(f"✔ {N_CLASSIFIERS} classifiers generated: "
      f"{len(sims_df)} items × {len(sims_df.columns) - 1} classifiers")
# ******************************************

# ******************************************
# Write to Excel (_hetero.xlsx)
# ******************************************
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill

hetero_path = EXCEL_PATH.parent / (
    EXCEL_PATH.stem.replace("_synthetic", "") + "_hetero.xlsx"
)

wb = load_workbook(EXCEL_PATH)
if SH_SIMS in wb.sheetnames:
    del wb[SH_SIMS]
ws = wb.create_sheet(SH_SIMS)

header_fill = PatternFill("solid", fgColor="FFF2CC")
header_font = Font(bold=True)
for col_idx, col_name in enumerate(sims_df.columns, start=1):
    cell = ws.cell(row=1, column=col_idx, value=col_name)
    cell.fill = header_fill
    cell.font = header_font

for row_idx, row in enumerate(sims_df.itertuples(index=False), start=2):
    for col_idx, value in enumerate(row, start=1):
        ws.cell(row=row_idx, column=col_idx, value=value)

wb.save(hetero_path)
EXCEL_PATH = hetero_path
print(f"✔ Simulations written to '{EXCEL_PATH.name}' "
      f"— original preserved — proceed to module-04")
# ******************************************


✔ corpus_df matches Spec: 130 rows — reusing.
Corpus sheet found: 130 rows — loaded as-is.
Corpus ready: 130 items | prevalence = 0.200
Classifier profiles assigned:
  lenient_01                 p_pos=0.750
  strict_02                  p_pos=0.250
  random_03                  p_pos=0.500
  prevalence_04              p_pos=0.200
  biased_pos_05              p_pos=0.900
  biased_neg_06              p_pos=0.100
  lenient_07                 p_pos=0.750
  strict_08                  p_pos=0.250
  random_09                  p_pos=0.500
  prevalence_10              p_pos=0.200
  biased_pos_11              p_pos=0.900
  biased_neg_12              p_pos=0.100
  lenient_13                 p_pos=0.750
  strict_14                  p_pos=0.250
  random_15                  p_pos=0.500
  prevalence_16              p_pos=0.200
  biased_pos_17              p_pos=0.900
  biased_neg_18              p_pos=0.100
  lenient_19                 p_pos=0.750
  strict_20                  p_pos=0.250
✔ 20 classifie

### module-99-requirements

In [ ]:
# module-99-requirements
# Writes a requirements.txt snapshot of the current environment.
# Run at the end of any session to document dependency versions.

import subprocess, pathlib

req_path = pathlib.Path.cwd() / "requirements.txt"
with open(req_path, "w") as fh:
    subprocess.run(["pip", "freeze"], stdout=fh, check=True)

print(f"✔ requirements.txt written to {req_path}")
